# losses

> Region-overlap and boundary losses for segmentation, written once and valid for both binary (`C==1`, sigmoid) and multi-class (`C>1`, softmax) heads.

A single helper converts raw logits into per-class probabilities and one-hot targets, so every loss below is class-count agnostic. Class-imbalance is handled by Tversky (tunable false-negative vs false-positive weighting); global overlap by Dice; thin-structure edges by a boundary-weighted cross-entropy.

In [ ]:
#| default_exp losses

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict

import torch
import torch.nn as nn
import torch.nn.functional as F

## Logits -> probabilities + one-hot targets

The one place that knows about binary vs multi-class. Everything downstream sees `(B, C, H, W)` probabilities and a matching `(B, C, H, W)` one-hot target.

In [ ]:
# experiment: shapes for a 2-class softmax case
logits = torch.randn(2, 3, 8, 8)        # B=2, C=3 classes
labels = torch.randint(0, 3, (2, 8, 8)) # integer labels
probs = torch.softmax(logits, 1)
onehot = F.one_hot(labels, 3).permute(0, 3, 1, 2).float()
print('probs', tuple(probs.shape), '| onehot', tuple(onehot.shape))

In [ ]:
#| export
def probs_and_targets(
    logits: torch.Tensor,    # (B, C, H, W) raw logits
    targets: torch.Tensor,   # (B, H, W) labels, or (B, 1, H, W)/(B, C, H, W) for binary/one-hot
) -> tuple:
    """Return `(probs, onehot)` both shaped `(B, C, H, W)`.

    `C == 1` uses sigmoid and treats `targets` as `{0,1}`; `C > 1` uses softmax
    and one-hot encodes integer-label targets.
    """
    c = logits.shape[1]
    if c == 1:
        probs = torch.sigmoid(logits)
        t = targets.float()
        if t.dim() == logits.dim() - 1:
            t = t.unsqueeze(1)
        return probs, t
    probs = torch.softmax(logits, dim=1)
    if targets.dim() == logits.dim():
        return probs, targets.float()
    onehot = F.one_hot(targets.long(), c).permute(0, 3, 1, 2).float()
    return probs, onehot

In [ ]:
# test
p, t = probs_and_targets(torch.randn(2, 1, 4, 4), torch.zeros(2, 4, 4))
assert p.shape == t.shape == (2, 1, 4, 4)
p, t = probs_and_targets(torch.randn(2, 3, 4, 4), torch.randint(0, 3, (2, 4, 4)))
assert p.shape == t.shape == (2, 3, 4, 4)

## Tversky-focal loss

Tversky generalises Dice with separate weights for false negatives (`alpha`) and false positives (`beta`). Raise `alpha` to punish missed targets (recall); raise `beta` to punish false alarms. The focal exponent `1/gamma` focuses on the hardest classes. We reduce per-class then average, so it is correct for any `C`.

In [ ]:
#| export
class TverskyFocalLoss(nn.Module):
    """Per-class Tversky-focal loss for binary or multi-class segmentation.

    Example:
        >>> loss = TverskyFocalLoss()(torch.randn(2, 1, 8, 8), torch.zeros(2, 8, 8))
        >>> loss.ndim
        0
    """

    def __init__(
        self,
        alpha: float = 0.7,   # false-negative weight (recall); higher -> fewer misses
        beta: float = 0.3,    # false-positive weight (precision); higher -> fewer false alarms
        gamma: float = 1.33,  # focal exponent (1/gamma); higher -> focus on hard classes
        smooth: float = 1.0,  # numerical stabiliser
    ):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p, t = probs_and_targets(logits, targets)
        dims = (0, 2, 3)
        tp = (p * t).sum(dims)
        fn = (t * (1 - p)).sum(dims)
        fp = ((1 - t) * p).sum(dims)
        tversky = (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)
        return ((1.0 - tversky) ** (1.0 / self.gamma)).mean()

## Dice loss

Soft Dice gives a stable global-overlap signal that complements Tversky.

In [ ]:
#| export
class DiceLoss(nn.Module):
    "Per-class soft Dice loss (binary or multi-class)."

    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p, t = probs_and_targets(logits, targets)
        dims = (0, 2, 3)
        inter = (p * t).sum(dims)
        dice = (2 * inter + self.smooth) / (p.sum(dims) + t.sum(dims) + self.smooth)
        return 1.0 - dice.mean()

## Boundary loss

Up-weights cross-entropy near target edges, where thin structures are most often lost. The boundary band is found on the GPU with `max_pool2d` (dilation) and its complement (erosion), avoiding a CPU round-trip.

In [ ]:
#| export
class BoundaryLoss(nn.Module):
    "Boundary-weighted cross-entropy (binary or multi-class)."

    def __init__(self, theta0: float = 3.0, theta: float = 5.0):
        super().__init__()
        self.theta0, self.theta = theta0, theta

    def _weight_map(self, fg: torch.Tensor) -> torch.Tensor:
        "fg: (B, 1, H, W) float {0,1}. Returns per-pixel weights."
        k = int(2 * self.theta0 + 1); pad = k // 2
        dilated = F.max_pool2d(fg, k, 1, pad)
        eroded = 1.0 - F.max_pool2d(1.0 - fg, k, 1, pad)
        boundary = (dilated - eroded).clamp(0, 1)
        return torch.where(boundary > 0, self.theta, 1.0)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        c = logits.shape[1]
        if c == 1:
            t = targets.float()
            if t.dim() == logits.dim() - 1:
                t = t.unsqueeze(1)
            per_px = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
            fg = t
        else:
            labels = targets.long()
            per_px = F.cross_entropy(logits, labels, reduction='none').unsqueeze(1)
            fg = (labels > 0).float().unsqueeze(1)
        return (per_px * self._weight_map(fg)).mean()

In [ ]:
#| export
class BCELoss(nn.Module):
    """Plain binary cross-entropy for segmentation (binary or multi-class).

    Use instead of `TverskyFocalLoss` when foreground is large relative to the
    image (e.g. a chip fills most of the field of view) and the class-imbalance
    correction of Tversky-focal is not needed.

    Example:
        >>> loss = BCELoss()(torch.randn(2, 1, 8, 8), torch.zeros(2, 8, 8))
        >>> loss.ndim
        0
    """

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        c = logits.shape[1]
        if c == 1:
            t = targets.float()
            if t.dim() == logits.dim() - 1:
                t = t.unsqueeze(1)
            return F.binary_cross_entropy_with_logits(logits, t)
        return F.cross_entropy(logits, targets.long())


`BCELoss` wraps `F.binary_cross_entropy_with_logits` (binary) and `F.cross_entropy` (multi-class) behind the same interface as the other loss components.

**When to use it:** set `w_bce > 0, w_tversky=0` when the foreground occupies a large fraction of the image — for example a rectangular or rotated-rectangular chip that fills most of the field of view. In that case the class-imbalance correction inside Tversky-focal is unnecessary and plain BCE gives a cleaner gradient signal. The typical pairing is `SegLoss(w_tversky=0, w_dice=0.5, w_bce=0.5)`.

**Test verifies:** scalar output, non-negativity, and gradient flow for binary (C=1) and multi-class (C=3) inputs.


In [ ]:
# test BCELoss: scalar, non-negative, backprops for binary and multi-class
for c, tgt in [(1, torch.zeros(2, 8, 8)), (3, torch.randint(0, 3, (2, 8, 8)))]:
    logits = torch.randn(2, c, 8, 8, requires_grad=True)
    loss = BCELoss()(logits, tgt)
    assert loss.ndim == 0 and float(loss) >= 0, f'C={c}: unexpected loss'
    loss.backward()
    assert logits.grad is not None
print('BCELoss ✓  binary + multi-class')


# Foreground-aware loss

Neuro-JEPA style foreground-aware loss for void detection

In [ ]:
#| export
class ForeGroundAwareLoss:
    
  def __init__(
    self, beta: float=0.1 # weight for background token (0.1 from Neuro-JEPA)
    ):
    self.beta = beta

  def __call__(
            self,
            z: torch.Tensor, # (B,K,D) predicted pixel embeddings for masked token
            h  : torch.Tensor, # (B,K,D) target pixel embeddings for masked token
            chip_mask: torch.Tensor, # (B,1,H,W) Original x-ray intensity
            masked_indices: torch.Tensor, # (B,K) indices of masked tokens
            patch_size: int=16, # patch size for pooling
            ) -> torch.Tensor:
    "Foreground-weighted cross-entropy (binary or multi-class)."
    p2 = torch.quantile(x_ray_patch, 0.02, dim=[2, 3], keepdim=True) # (B,1,1,1)
    p98 = torch.quantile(x_ray_patch, 0.98, dim=[2, 3], keepdim=True) # (B,1,1,1)
    
    # Foreground = within normal solder range,
    # for void detection: void pixels are foreground, background is the region outside chip entirely

    patch_mask = F.avg_pool2d(chip_mask.float(), kernel_size=patch_size, stride=patch_size)

    patch_mask_flat= patch_mask.flatten(1) # (B, num_patches)

    token_fg = torch.gather(
        patch_mask_flat, 1, masked_indices.long()
    )


    # Weight: foreground=1.0, background=beta (0.1 from Neuro-JEPA)
    w = torch.where(
        token_fg > 0.5,
        torch.ones_like(token_fg),
        torch.full_like(token_fg, self.beta)
    
    )

    l1 = torch.abs(z - h).mean(dim=-1) # (B, K)
    loss = (w*l1).sum() / (w.sum() + 1e-8) # weighted average
    return loss

## Combined loss with a phase schedule

The total loss is a weighted sum. Boundary refinement is noisy early on, so a phase schedule starts with coarse shape (Tversky+Dice) and adds boundary weight later.

In [ ]:
#| export
class SegLoss(nn.Module):
    """Weighted Tversky-focal + Dice + Boundary + BCE loss returning a breakdown dict.

    Call `set_phase(1|2|3)` at epoch boundaries to anneal the boundary weight.
    Set `w_bce > 0, w_tversky=0` for a BCE + Dice schedule when foreground is
    large and class-imbalance correction is not needed.
    """

    def __init__(
        self,
        alpha: float = 0.7, beta: float = 0.3, gamma: float = 1.33,  # Tversky params
        w_tversky: float = 0.5, w_dice: float = 0.5, w_boundary: float = 0.0,  # component weights
        w_bce: float = 0.0,  # plain BCE weight; set > 0 to enable
    ):
        super().__init__()
        self.w_tversky, self.w_dice, self.w_boundary, self.w_bce = w_tversky, w_dice, w_boundary, w_bce
        self.tversky = TverskyFocalLoss(alpha, beta, gamma)
        self.dice = DiceLoss()
        self.boundary = BoundaryLoss()
        self.bce = BCELoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> Dict[str, torch.Tensor]:
        l_tv = self.tversky(logits, targets)
        l_dc = self.dice(logits, targets)
        l_bd = self.boundary(logits, targets) if self.w_boundary > 0 else torch.zeros((), device=logits.device)
        l_bce = self.bce(logits, targets) if self.w_bce > 0 else torch.zeros((), device=logits.device)
        total = self.w_tversky * l_tv + self.w_dice * l_dc + self.w_boundary * l_bd + self.w_bce * l_bce
        return {'loss': total, 'loss_tversky': l_tv.detach(), 'loss_dice': l_dc.detach(),
                'loss_boundary': l_bd.detach(), 'loss_bce': l_bce.detach()}

    def set_phase(self, phase: int) -> None:
        "Phase 1: coarse shape; 2: add boundary; 3: polish edges."
        weights = {1: (0.6, 0.4, 0.0), 2: (0.5, 0.3, 0.2), 3: (0.4, 0.3, 0.3)}
        self.w_tversky, self.w_dice, self.w_boundary = weights[phase]

    def set_weights(self, w_tversky: float, w_dice: float, w_boundary: float, w_bce: float = 0.0) -> None:
        "Set component weights directly (used by configurable phase schedules)."
        self.w_tversky, self.w_dice, self.w_boundary, self.w_bce = w_tversky, w_dice, w_boundary, w_bce


#| export
def resolve_loss_weights(
    epoch: int,
    phases,
) -> tuple:
    """Return `(w_tversky, w_dice, w_boundary, w_bce)` for `epoch` from a phase list.

    `w_bce` defaults to `0.0` when absent from a phase entry, so existing configs
    that only specify `w_tversky/w_dice/w_boundary` continue to work unchanged.
    """
    for phase in phases:
        until = phase.get('until_epoch', -1)
        if until is None or until < 0 or epoch < until:
            return (float(phase['w_tversky']), float(phase['w_dice']),
                    float(phase['w_boundary']), float(phase.get('w_bce', 0.0)))
    last = phases[-1]
    return (float(last['w_tversky']), float(last['w_dice']),
            float(last['w_boundary']), float(last.get('w_bce', 0.0)))


In [ ]:
# test: every loss is a non-negative scalar and backprops, for binary AND multi-class
for c, tgt in [(1, torch.zeros(2, 16, 16)), (3, torch.randint(0, 3, (2, 16, 16)))]:
    logits = torch.randn(2, c, 16, 16, requires_grad=True)
    crit = SegLoss(w_tversky=0.5, w_dice=0.4, w_boundary=0.1)
    out = crit(logits, tgt)
    assert out['loss'].ndim == 0 and float(out['loss']) >= 0
    assert 'loss_bce' in out and float(out['loss_bce']) == 0.0  # w_bce=0 → zero
    out['loss'].backward()
    assert logits.grad is not None
crit.set_phase(3)
assert crit.w_boundary == 0.3

# BCE + Dice mode: useful when foreground is large (e.g. chip fills the image)
logits = torch.randn(2, 1, 16, 16, requires_grad=True)
crit_bce = SegLoss(w_tversky=0.0, w_dice=0.5, w_boundary=0.0, w_bce=0.5)
out = crit_bce(logits, torch.zeros(2, 16, 16))
assert float(out['loss']) > 0 and float(out['loss_bce']) > 0
out['loss'].backward()
assert logits.grad is not None

# resolve_loss_weights returns 4-tuple; w_bce defaults to 0 for legacy configs
phases = [
    {'until_epoch': 10, 'w_tversky': 0.6, 'w_dice': 0.4, 'w_boundary': 0.0},
    {'until_epoch': -1, 'w_tversky': 0.0, 'w_dice': 0.5, 'w_boundary': 0.0, 'w_bce': 0.5},
]
assert resolve_loss_weights(5, phases) == (0.6, 0.4, 0.0, 0.0)
assert resolve_loss_weights(15, phases) == (0.0, 0.5, 0.0, 0.5)
print('SegLoss ✓  Tversky+Dice+Boundary | BCE+Dice | phase schedule')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()